# EXP1 — Верификация (full, день 2)

Репрезентативная выборка из 83 сгенерированных сценариев (15 happy-path + 68 с ровно одним инъецированным нарушением из семи типов SV*). Каждый сценарий детерминированно создаётся `_common/generator.py` с `random.seed=42`, ground-truth задаётся в `sweep.build_cases()`.

Pilot дня 1 на 8 готовых сценариях — `notebook_pilot_day1.ipynb` (сохраняется для сравнения). Методология та же: confusion matrix per-property × per-scenario, positive class — обнаруженное нарушение.

In [1]:
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
EXPERIMENTS_ROOT = NOTEBOOK_DIR.parent
PROJECT_ROOT = EXPERIMENTS_ROOT.parent
BACKEND_SRC = PROJECT_ROOT / 'code' / 'backend' / 'src'
SCENARIOS_DIR = NOTEBOOK_DIR / 'scenarios'
RESULTS_DIR = NOTEBOOK_DIR / 'results'
PZ_FIGURES_DIR = PROJECT_ROOT / 'pz' / 'figures' / 'exp1'

sys.path.insert(0, str(BACKEND_SRC))
sys.path.insert(0, str(EXPERIMENTS_ROOT))
sys.path.insert(0, str(NOTEBOOK_DIR))

from owlready2 import World

from services.ontology_core import OntologyCore
from services.cache_manager import CacheManager
from services.reasoning import ReasoningOrchestrator
from services.verification import VerificationService

from _common.generator import generate_scenario
from _common.metrics import (
    aggregate_by_property,
    format_csv,
    format_markdown_table,
    macro_average,
)

import sweep as sweep_mod

SCENARIOS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
PZ_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

cases = sweep_mod.build_cases()
print(f'Сценариев в выборке: {len(cases)}')

Сценариев в выборке: 83


In [2]:
gen_start = time.perf_counter()
generated: list[tuple[sweep_mod.SweepCase, Path]] = []
for case in cases:
    owl_path = SCENARIOS_DIR / f"{case.name}.owl"
    generate_scenario(case.config, owl_path)
    generated.append((case, owl_path))
gen_duration = time.perf_counter() - gen_start
print(f'Сгенерировано OWL: {len(generated)} за {gen_duration:.1f} с')

Сгенерировано OWL: 83 за 1.7 с


In [3]:
def run_verify(owl_path: Path, course_id: str, include_subsumption: bool):
    world = World()
    onto = world.get_ontology(f"file://{str(owl_path).replace(os.sep, '/')}").load()
    core = OntologyCore(onto_path=str(owl_path), world=world)
    core.onto = onto
    core.world = world
    reasoner = ReasoningOrchestrator(core.onto)
    service = VerificationService(core, reasoner=reasoner, cache=CacheManager(None))
    t0 = time.perf_counter()
    report = service.verify(course_id, include_subsumption=include_subsumption)
    return report, (time.perf_counter() - t0) * 1000.0

runs: list[tuple[sweep_mod.SweepCase, object, float]] = []
verify_start = time.perf_counter()
for case, owl_path in generated:
    report, duration_ms = run_verify(owl_path, case.config.course_id, case.include_subsumption)
    runs.append((case, report, duration_ms))
verify_duration = time.perf_counter() - verify_start
print(f'Прогнано через VerificationService: {len(runs)} за {verify_duration:.1f} с')

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib

* Owlready2 * Pellet took 1.14985990524292 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\gal

* Owlready2 * Pellet took 1.0520288944244385 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready2 * Pellet took 1.0531036853790283 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready2 * Pellet took 1.047593355178833 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\ga

...

* Owlready * Adding relation sv5_07_t90.gen_student_advanced satisfies sv5_07_t90.gen_subj_group_sub_group


* Owlready2 * Pellet took 1.0437901020050049 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready * Adding relation sv5_08_t55.gen_student_advanced satisfies sv5_08_t55.gen_subj_group_sub_group


* Owlready * Adding relation sv5_09_t40.gen_student_advanced satisfies sv5_09_t40.gen_subj_group_sub_group
Прогнано через VerificationService: 83 за 93.1 с


* Owlready2 * Pellet took 1.080512285232544 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [4]:
pairs = [(dict(report.properties), case.expected) for case, report, _ in runs]
matrices = aggregate_by_property(pairs)
macro = macro_average(matrices)
md_table = format_markdown_table(matrices)
csv_text = format_csv(matrices)
print(md_table)
print()
print(f"macro-avg: precision={macro['precision']:.3f}  recall={macro['recall']:.3f}  f1={macro['f1']:.3f}")

| Свойство | TP | FP | FN | TN | Precision | Recall | F1 | Support |
|---|---|---|---|---|---|---|---|---|
| consistency | 10 | 0 | 0 | 15 | 1.000 | 1.000 | 1.000 | 10 |
| acyclicity | 10 | 0 | 0 | 15 | 1.000 | 1.000 | 1.000 | 10 |
| reachability | 28 | 0 | 0 | 15 | 1.000 | 1.000 | 1.000 | 28 |
| redundancy | 10 | 0 | 0 | 0 | 1.000 | 1.000 | 1.000 | 10 |
| subsumption | 10 | 0 | 0 | 0 | 1.000 | 1.000 | 1.000 | 10 |
| **macro-avg** | . | . | . | . | 1.000 | 1.000 | 1.000 | . |

macro-avg: precision=1.000  recall=1.000  f1=1.000


In [5]:
# Разрез по типу дефекта: сколько сценариев каждого класса распознано корректно.
import collections

bucket = collections.defaultdict(lambda: {'total': 0, 'correct': 0})
for case, report, _ in runs:
    fault = case.config.fault or 'happy'
    bucket[fault]['total'] += 1
    correct = True
    for prop, exp_status in case.expected.items():
        actual = report.properties.get(prop)
        if actual is None or actual.status != exp_status:
            correct = False
            break
    if correct:
        bucket[fault]['correct'] += 1

breakdown_lines = ['| Класс | Сценариев | Распознано | Accuracy |', '|---|---|---|---|']
for fault in sorted(bucket):
    total = bucket[fault]['total']
    correct = bucket[fault]['correct']
    acc = correct / total if total else float('nan')
    breakdown_lines.append(f'| {fault} | {total} | {correct} | {acc:.3f} |')
breakdown_md = '\n'.join(breakdown_lines)
print(breakdown_md)

| Класс | Сценариев | Распознано | Accuracy |
|---|---|---|---|
| happy | 15 | 15 | 1.000 |
| sv1_disjointness | 10 | 10 | 1.000 |
| sv2_cycle | 10 | 10 | 1.000 |
| sv3_atomic_threshold | 10 | 10 | 1.000 |
| sv3_empty_date | 8 | 8 | 1.000 |
| sv3_structural | 10 | 10 | 1.000 |
| sv4_redundant | 10 | 10 | 1.000 |
| sv5_subject | 10 | 10 | 1.000 |


In [6]:
durations_rows = ['scenario,fault,include_subsumption,duration_ms']
for case, _, duration_ms in runs:
    durations_rows.append(
        f"{case.name},{case.config.fault or 'happy'},{case.include_subsumption},{duration_ms:.1f}"
    )
durations_csv = '\n'.join(durations_rows) + '\n'

summary_md = md_table + '\n\n## Breakdown по типам\n\n' + breakdown_md + '\n'
summary_md += f"\n## Параметры\n\n- Сценариев: {len(runs)}\n"
summary_md += f"- Генерация всего: {gen_duration:.1f} с\n"
summary_md += f"- Верификация всего: {verify_duration:.1f} с\n"
summary_md += f"- Средняя верификация на сценарий: {verify_duration * 1000.0 / len(runs):.0f} ms\n"

for target_dir in (RESULTS_DIR, PZ_FIGURES_DIR):
    (target_dir / 'full_day2.csv').write_text(csv_text, encoding='utf-8')
    (target_dir / 'full_day2.md').write_text(summary_md, encoding='utf-8')
    (target_dir / 'full_day2_durations.csv').write_text(durations_csv, encoding='utf-8')

print('Экспортировано:')
print(f'  {RESULTS_DIR}')
print(f'  {PZ_FIGURES_DIR}')

Экспортировано:
  C:\Documents\itmo\vkr\vkr-access-control\experiments\exp1_verification\results
  C:\Documents\itmo\vkr\vkr-access-control\pz\figures\exp1
